<a href="https://colab.research.google.com/github/rajeev198886/SKS-Data-Science-Internship-2026/blob/main/Churn_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.model_selection import GridSearchCV

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Telco_Customer_Churn_Dataset.csv.csv to Telco_Customer_Churn_Dataset.csv (1).csv


In [3]:
data = pd.read_csv("Telco_Customer_Churn_Dataset.csv (1).csv")

data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

In [5]:
data["TotalCharges"].fillna(
    data["TotalCharges"].mean(),
    inplace=True
)

/tmp/ipykernel_2057/2034971299.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["TotalCharges"].fillna(


In [6]:
data.drop("customerID", axis=1, inplace=True)

In [7]:
encoder = LabelEncoder()

for column in data.columns:
    if data[column].dtype == "object":
        data[column] = encoder.fit_transform(data[column])

In [8]:
X = data.drop("Churn", axis=1)

y = data["Churn"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [10]:
selector = SelectKBest(score_func=chi2, k=10)

X_train_selected = selector.fit_transform(
    X_train,
    y_train
)

X_test_selected = selector.transform(
    X_test
)

print("Feature Selection Completed")

Feature Selection Completed


In [11]:
lr = LogisticRegression(max_iter=1000)

lr.fit(
    X_train_selected,
    y_train
)

lr_prediction = lr.predict(X_test_selected)

In [12]:
print("Accuracy :", accuracy_score(y_test, lr_prediction))

print("Precision :", precision_score(y_test, lr_prediction))

print("Recall :", recall_score(y_test, lr_prediction))

print("F1 Score :", f1_score(y_test, lr_prediction))

Accuracy : 0.808374733853797
Precision : 0.6634920634920635
Recall : 0.5603217158176944
F1 Score : 0.6075581395348837


In [13]:
dt = DecisionTreeClassifier()

dt.fit(
    X_train_selected,
    y_train
)

dt_prediction = dt.predict(X_test_selected)

In [14]:
print("Accuracy :", accuracy_score(y_test, dt_prediction))

print("Precision :", precision_score(y_test, dt_prediction))

print("Recall :", recall_score(y_test, dt_prediction))

print("F1 Score :", f1_score(y_test, dt_prediction))

Accuracy : 0.7437899219304471
Precision : 0.5163934426229508
Recall : 0.5067024128686327
F1 Score : 0.5115020297699594


In [15]:
print(classification_report(
    y_test,
    lr_prediction
))

              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1036
           1       0.66      0.56      0.61       373

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [16]:
print(confusion_matrix(
    y_test,
    lr_prediction
))

[[930 106]
 [164 209]]


In [17]:
parameters = {
    "max_depth":[3,5,7,10],
    "criterion":["gini","entropy"]
}

grid = GridSearchCV(
    DecisionTreeClassifier(),
    parameters,
    cv=5
)

grid.fit(
    X_train_selected,
    y_train
)

print(grid.best_params_)

{'criterion': 'entropy', 'max_depth': 5}


In [18]:
best_model = grid.best_estimator_

prediction = best_model.predict(
    X_test_selected
)

print("Accuracy :", accuracy_score(y_test,prediction))

print(classification_report(
    y_test,
    prediction
))

Accuracy : 0.7885024840312278
              precision    recall  f1-score   support

           0       0.82      0.92      0.86      1036
           1       0.65      0.43      0.52       373

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.69      1409
weighted avg       0.77      0.79      0.77      1409



In [19]:
comparison = pd.DataFrame({

    "Model":[
        "Logistic Regression",
        "Decision Tree"
    ],

    "Accuracy":[
        accuracy_score(y_test,lr_prediction),
        accuracy_score(y_test,dt_prediction)
    ]

})

comparison

,Model,Accuracy
0,Logistic Regression,0.808375
1,Decision Tree,0.743790


In [20]:
output = X_test.copy()

output["Actual"] = y_test

output["Prediction"] = prediction

output.to_csv(
    "Churn_Prediction_Output.csv",
    index=False
)

print("Prediction File Saved")

Prediction File Saved
